In [ ]:
# Install dependencies

%pip install claude-agent-sdk

In [ ]:
import json
from claude_agent_sdk import query, ClaudeAgentOptions

# Execute claude code from a jupyter notebook

Do not remove the manual lopp code, even if sonnet tells you.
if so, the code would result in "claude code has reached max turns" error 

In [ ]:
import asyncio
import sys
import threading

def run_async(coro, timeout=300):
    """Run async code in a background thread with an explicit ProactorEventLoop.

    Two problems in Jupyter on Windows:
    1. asyncio.run() fails because Jupyter already has a running event loop.
    2. The Jupyter event loop is a SelectorEventLoop which cannot spawn subprocesses.
       ProactorEventLoop is required for subprocess I/O (used by the Agent SDK).
    Using an explicit loop avoids anyio.run()'s loop_factory path, which can hang on Windows.
    Each call gets a fully isolated loop so sequential calls don't bleed sniffio context.
    """
    result_container = [None]
    error_container = [None]

    async def _wrapper():
        result_container[0] = await coro

    def thread_fn():
        if sys.platform == "win32":
            loop = asyncio.ProactorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(_wrapper())
        except Exception as e:
            error_container[0] = e
        finally:
            loop.close()
            asyncio.set_event_loop(None)

    t = threading.Thread(target=thread_fn, daemon=True)
    t.start()
    t.join(timeout=timeout)

    if t.is_alive():
        raise TimeoutError(f"Async operation timed out after {timeout} seconds")

    if error_container[0] is not None:
        raise error_container[0]
    return result_container[0]

In [ ]:
async def _messages_to_async_iterable(messages):
    session_id = "session"
    for message in messages:
        if message["role"] == "user":
            yield {
                "type": "user",
                "message": {"role": "user", "content": message["content"]},
                "parent_tool_use_id": None,
                "session_id": session_id,
            }


def generate_prompt(messages):
    if len(messages) == 1 and messages[0]["role"] == "user":
        return messages[0]["content"]
    return _messages_to_async_iterable(messages)

In [ ]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


async def chat(messages):
    prompt = generate_prompt(messages)

    options = ClaudeAgentOptions(
        model="claude-sonnet-4-6",
        max_turns=1,
        allowed_tools=[],
    )

    result = None
    async for message in query(prompt=prompt, options=options):
        if hasattr(message, "result"):
            result = message.result
    return result

In [ ]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = run_async(chat(messages))
    if "```" in eval_text:
        eval_text = eval_text.split("```json")[-1].split("```")[0].strip()
    return json.loads(eval_text)

In [ ]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = run_async(chat(messages))
    return output

In [ ]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [ ]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
import json

with open("010/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))